In [1]:
# !pip install pandas numpy rapidfuzz lightgbm optuna tqdm

import os
import re
import gc
import time
from collections import defaultdict
import pandas as pd
import numpy as np
import rapidfuzz
import lightgbm as lgb
import optuna
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- CONFIGURATION ---
DATA_DIR = "dataset/"
OUTPUT_DIR = "output/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
LEGAL_SUFFIXES = {
    "incorporated", "inc", "corporation", "corp",
    "limited", "ltd", "llc", "llp",
    "private", "pvt", "company", "co",
}

def clean_text(text):
    if pd.isna(text): return ""
    text = re.sub(r'\b(null|nan|none|na|n/a|undefined)\b', ' ', str(text), flags=re.IGNORECASE)
    text = str(text).lower()
    text = text.replace("&", " and ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_name(text):
    s = clean_text(text)
    tokens = [t for t in s.split() if t not in LEGAL_SUFFIXES]
    return " ".join(tokens)

def sorted_tokens(text):
    return " ".join(sorted(set(text.split())))

def preprocess_dataframe(df):
    """Generates multiple representations to maximize hash-blocking recall."""
    df = df.copy()
    if 'country' in df.columns:
        df['country_norm'] = df['country'].astype(str).str.strip().str.lower()
        df['country_norm'] = df['country_norm'].replace('nan', 'unknown')
    else:
        df['country_norm'] = 'unknown'

    df['name_norm'] = df['business_name'].apply(normalize_name)
    df['addr_norm'] = df['business_address'].apply(clean_text)
    
    # Compact versions (no spaces)
    df['name_compact'] = df['name_norm'].str.replace(" ", "")
    df['addr_compact'] = df['addr_norm'].str.replace(" ", "")
    
    # Sorted token versions (handles out-of-order words)
    df['name_sorted'] = df['name_norm'].apply(sorted_tokens)
    df['full_text'] = df['name_norm'] + " " + df['addr_norm']
    
    return df

In [3]:
def build_indexes(df):
    """Builds O(N) hash maps for instantaneous candidate lookups."""
    indexes = {
        "name_key": defaultdict(list),
        "addr_key": defaultdict(list),
        "token_key": defaultdict(list)
    }
    
    for row in df.itertuples(index=False):
        c = row.country_norm
        if row.name_compact:
            indexes["name_key"][f"{c}|{row.name_compact}"].append(row.entity_id)
        if row.addr_compact:
            indexes["addr_key"][f"{c}|{row.addr_compact}"].append(row.entity_id)
        if row.name_sorted:
            indexes["token_key"][f"{c}|{row.name_sorted}"].append(row.entity_id)
            
    return indexes

def get_candidates(s1_df, indexes):
    """Retrieves candidates using pre-built indexes."""
    results = {'source1_entity_id': [], 'candidate_entity_id': []}
    
    for row in tqdm(s1_df.itertuples(index=False), total=len(s1_df), desc="Fetching Candidates"):
        cands = set()
        c = row.country_norm
        
        if row.name_compact: cands.update(indexes["name_key"].get(f"{c}|{row.name_compact}", []))
        if row.addr_compact: cands.update(indexes["addr_key"].get(f"{c}|{row.addr_compact}", []))
        if row.name_sorted: cands.update(indexes["token_key"].get(f"{c}|{row.name_sorted}", []))
            
        # Fast bulk append
        if cands:
            results['source1_entity_id'].extend([row.entity_id] * len(cands))
            results['candidate_entity_id'].extend(list(cands))
            
    return pd.DataFrame(results)

In [4]:
def engineer_features(pairs_df, s1_dict, cand_dict):
    n_pairs = len(pairs_df)
    print(f"Engineering features for {n_pairs} candidate pairs...")
    
    # Pre-allocate memory-efficient 32-bit float arrays
    f_name_jaro = np.zeros(n_pairs, dtype=np.float32)
    f_name_token = np.zeros(n_pairs, dtype=np.float32)
    f_addr_jaro = np.zeros(n_pairs, dtype=np.float32)
    f_addr_token = np.zeros(n_pairs, dtype=np.float32)
    f_full_token = np.zeros(n_pairs, dtype=np.float32)
    
    for i, row in enumerate(tqdm(pairs_df.itertuples(index=False), total=n_pairs, desc="Extracting Metrics")):
        s1 = s1_dict[row.source1_entity_id]
        cand = cand_dict[row.candidate_entity_id]
        
        f_name_jaro[i] = rapidfuzz.distance.JaroWinkler.normalized_similarity(s1['name_norm'], cand['name_norm'])
        f_name_token[i] = rapidfuzz.fuzz.token_set_ratio(s1['name_norm'], cand['name_norm']) / 100.0
        f_addr_jaro[i] = rapidfuzz.distance.JaroWinkler.normalized_similarity(s1['addr_norm'], cand['addr_norm'])
        f_addr_token[i] = rapidfuzz.fuzz.token_set_ratio(s1['addr_norm'], cand['addr_norm']) / 100.0
        f_full_token[i] = rapidfuzz.fuzz.token_set_ratio(s1['full_text'], cand['full_text']) / 100.0
        
    # Assign arrays directly to avoid copying the dataframe
    pairs_df['name_jaro'] = f_name_jaro
    pairs_df['name_token'] = f_name_token
    pairs_df['addr_jaro'] = f_addr_jaro
    pairs_df['addr_token'] = f_addr_token
    pairs_df['full_token'] = f_full_token
    
    return pairs_df

def f05_macro(preds_dict, truths_dict):
    scores = []
    for s1 in truths_dict.keys():
        true_set = set(truths_dict.get(s1, []))
        pred_set = set(preds_dict.get(s1, []))
        
        if not true_set and not pred_set: scores.append(1.0)
        elif not true_set and pred_set: scores.append(0.0)
        elif true_set and not pred_set: scores.append(0.0)
        else:
            tp = len(true_set & pred_set)
            fp = len(pred_set - true_set)
            fn = len(true_set - pred_set)
            
            p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            
            if p == 0 or r == 0: scores.append(0.0)
            else: scores.append((1.25 * p * r) / (0.25 * p + r))
            
    return sum(scores) / len(scores) if scores else 0.0

In [5]:
print("Loading Datasets...")
train_s1 = preprocess_dataframe(pd.read_csv(f"{DATA_DIR}train/train_source1.tsv", sep="\t"))
train_s2 = preprocess_dataframe(pd.read_csv(f"{DATA_DIR}train/train_source2.tsv", sep="\t"))
train_s3 = preprocess_dataframe(pd.read_csv(f"{DATA_DIR}train/train_source3.tsv", sep="\t"))

gt_df = pd.read_csv(f"{DATA_DIR}train/train_ground_truth.tsv", sep="\t", dtype=str).fillna("")
gt_df['matched_entity_ids'] = gt_df['matched_entity_ids'].apply(lambda x: [v.strip() for v in x.split(",") if v.strip()])
gt_dict = gt_df.set_index('source1_entity_id')['matched_entity_ids'].to_dict()

gt_exploded = gt_df.explode('matched_entity_ids').dropna()

# 1. Create a minimal DataFrame of just the positive ground truth pairs
pos_df = gt_exploded[['source1_entity_id', 'matched_entity_ids']].rename(columns={'matched_entity_ids': 'candidate_entity_id'})
pos_df['target'] = np.int8(1)

train_s1_ids, val_s1_ids = train_test_split(train_s1['entity_id'].unique(), test_size=0.2, random_state=42)
s1_train_split = train_s1[train_s1['entity_id'].isin(train_s1_ids)].copy()
s1_val_split = train_s1[train_s1['entity_id'].isin(val_s1_ids)].copy()

cand_base_train = pd.concat([train_s2, train_s3], ignore_index=True)

print("\n--- Building Master Dictionaries (Once) ---")
train_indexes = build_indexes(cand_base_train)
cand_dict_train = cand_base_train.set_index('entity_id')[['name_norm', 'addr_norm', 'full_text']].to_dict('index')
s1_train_dict = s1_train_split.set_index('entity_id')[['name_norm', 'addr_norm', 'full_text']].to_dict('index')
s1_val_dict = s1_val_split.set_index('entity_id')[['name_norm', 'addr_norm', 'full_text']].to_dict('index')

print("\n--- Training Set Generation ---")
train_feat = get_candidates(s1_train_split, train_indexes)
train_feat = engineer_features(train_feat, s1_train_dict, cand_dict_train)

# 2. Vectorized Target Labeling: Left join the positive pairs, fill NaNs with 0
print("Assigning Target Labels (Vectorized Merge)...")
train_feat = train_feat.merge(pos_df, on=['source1_entity_id', 'candidate_entity_id'], how='left')
train_feat['target'] = train_feat['target'].fillna(0).astype(np.int8)

print("\n--- Validation Set Generation ---")
val_feat = get_candidates(s1_val_split, train_indexes)
val_feat = engineer_features(val_feat, s1_val_dict, cand_dict_train)

# 3. O(1) SET LOOKUP (Fixes the infinite CPU hang)
val_s1_set = set(val_s1_ids)
val_truth = {k: gt_dict[k] for k in val_s1_set if k in gt_dict}

feature_cols = ['name_jaro', 'name_token', 'addr_jaro', 'addr_token', 'full_token']

# Purge memory
print("Purging intermediate memory...")
del train_s1, train_s2, train_s3, cand_base_train, pos_df
del train_indexes, cand_dict_train, s1_train_dict, s1_val_dict
gc.collect()

print("\n✅ Cell 5 Execution Successfully Completed!")

Loading Datasets...

--- Building Master Dictionaries (Once) ---

--- Training Set Generation ---


Fetching Candidates:   0%|          | 0/1765456 [00:00<?, ?it/s]

Engineering features for 66743308 candidate pairs...


Extracting Metrics:   0%|          | 0/66743308 [00:00<?, ?it/s]

Assigning Target Labels (Vectorized Merge)...

--- Validation Set Generation ---


Fetching Candidates:   0%|          | 0/441365 [00:00<?, ?it/s]

Engineering features for 16910824 candidate pairs...


Extracting Metrics:   0%|          | 0/16910824 [00:01<?, ?it/s]

Purging intermediate memory...

✅ Cell 5 Execution Successfully Completed!


In [6]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'num_leaves': trial.suggest_int('num_leaves', 15, 50),
        'class_weight': 'balanced',
        'random_state': 42,
        'verbose': -1,
        'n_jobs': -1
    }
    
    gbm = lgb.LGBMClassifier(**params)
    gbm.fit(train_feat[feature_cols], train_feat['target'])
    val_probs = gbm.predict_proba(val_feat[feature_cols])[:, 1]
    
    best_f05_trial = 0.0
    for th in np.arange(0.50, 0.90, 0.05):
        mask = val_probs >= th
        preds_df = val_feat[mask]
        preds = preds_df.groupby('source1_entity_id')['candidate_entity_id'].apply(list).to_dict()
        
        score = f05_macro(preds, val_truth)
        if score > best_f05_trial:
            best_f05_trial = score
            
    return best_f05_trial

print("Running LightGBM Optuna Trials...")
study = optuna.create_study(direction='maximize', study_name="Business_ER_HashML")
study.optimize(objective, n_trials=15)

print(f"\n✅ Best Validation F0.5: {study.best_value:.4f}")

# Train the final model with best params
final_params = study.best_params
final_params.update({'class_weight': 'balanced', 'random_state': 42, 'verbose': -1, 'n_jobs': -1})

model = lgb.LGBMClassifier(**final_params)
model.fit(train_feat[feature_cols], train_feat['target'])

# Lock the final threshold
val_feat['prob'] = model.predict_proba(val_feat[feature_cols])[:, 1]
best_th, best_f05 = 0.5, 0.0

for th in np.arange(0.50, 0.95, 0.05):
    preds = val_feat[val_feat['prob'] >= th].groupby('source1_entity_id')['candidate_entity_id'].apply(list).to_dict()
    score = f05_macro(preds, val_truth)
    if score > best_f05:
        best_f05, best_th = score, th

print(f"✅ Final Threshold Locked: {best_th:.2f}")

del train_feat, val_feat
gc.collect()

Running LightGBM Optuna Trials...


[W 2026-09-26 18:09:34,111] Trial 1 failed with parameters: {'n_estimators': 244, 'learning_rate': 0.13039672177594294, 'max_depth': 5, 'num_leaves': 24} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/opt/conda/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_1247/1993296114.py", line 14, in objective
    gbm.fit(train_feat[feature_cols], train_feat['target'])
  File "/opt/conda/lib/python3.12/site-packages/lightgbm/sklearn.py", line 1560, in fit
    super().fit(
  File "/opt/conda/lib/python3.12/site-packages/lightgbm/sklearn.py", line 1049, in fit
    self._Booster = train(
                    ^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/lightgbm/engine.py", line 322, in train
    booster.update(fobj=fobj)
  File "/opt/conda/lib/python3.12/site-packages/lightgbm/basic.py", line 4155, in update
    _LIB.L

KeyboardInterrupt: 

In [ ]:
print("\n--- Test Set Inference ---")
test_s1 = preprocess_dataframe(pd.read_csv(f"{DATA_DIR}test/test_source1.tsv", sep="\t"))
test_s2 = preprocess_dataframe(pd.read_csv(f"{DATA_DIR}test/test_source2.tsv", sep="\t"))
test_s3 = preprocess_dataframe(pd.read_csv(f"{DATA_DIR}test/test_source3.tsv", sep="\t"))

test_cand_base = pd.concat([test_s2, test_s3], ignore_index=True)

test_blocks, test_cand_dict = get_candidates(test_s1, test_cand_base)
test_feat = engineer_features(test_blocks, test_s1, test_cand_dict)

# ML Predictions
test_feat['prob'] = model.predict_proba(test_feat[feature_cols])[:, 1]

# Aggregate matches and candidates
cands_dict = test_feat.groupby('source1_entity_id')['candidate_entity_id'].apply(lambda x: sorted(list(set(x)))).to_dict()
matches_dict = test_feat[test_feat['prob'] >= best_th].groupby('source1_entity_id')['candidate_entity_id'].apply(lambda x: sorted(list(set(x)))).to_dict()

# Guarantee exactly 1 row per Test S1 entity
cand_rows, match_rows = [], []
for s1_id in test_s1['entity_id'].values:
    c_str = ",".join(cands_dict.get(s1_id, []))
    m_str = ",".join(matches_dict.get(s1_id, []))
    cand_rows.append(f"{s1_id}\t{c_str}\n")
    match_rows.append(f"{s1_id}\t{m_str}\n")
    
# Save exports
with open(f"{OUTPUT_DIR}candidate_pairs.tsv", "w") as f:
    f.write("source1_entity_id\tcandidate_entity_ids\n")
    f.writelines(cand_rows)
    
with open(f"{OUTPUT_DIR}matching_results.tsv", "w") as f:
    f.write("source1_entity_id\tmatched_entity_ids\n")
    f.writelines(match_rows)
    
print("✅ Output Generated! Run `utils/validate_submission.py`.")